# Gen AI Project: 2
### NxtWave Academy | Build an HR chatbot using RAG

---

## Objective

Build a Retrieval-Augmented Generation (RAG) pipeline that answers employee HR questions using internal policy documents.

## What you will build

- Load and process HR policy documents
- Create chunks and embeddings
- Build a vector database using FAISS
- Implement a RAG pipeline with guardrails
- Generate your `submission.csv`


## Write your solution code Below

## Generate `submission.csv`

Generate your final `submission.csv` file for submission.

Do not modify this cell.


# install Dependancy 

In [84]:
import sys

print("Python being used:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python being used:
C:\ProgramData\anaconda3\python.exe

Python version:
3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]


In [85]:
import sys

!{sys.executable} -m pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface chromadb

Defaulting to user installation because normal site-packages is not writeable


In [86]:
print("Installing Dependencies")

%pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    langchain-google-genai \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface-hub \
    groq \
    langsmith \
    python-dotenv \
    tiktoken

print("Installation completed")

Installing Dependencies
Note: you may need to restart the kernel to use updated packages.
Installation completed


In [87]:
import langchain
import langchain_community
import langchain_huggingface
import langchain_groq
import langchain_google_genai
import langchain_openai
import faiss
import pypdf
import transformers
import torch
import groq

In [88]:
import sys

print(sys.executable)

C:\ProgramData\anaconda3\python.exe


In [ ]:
LLM_PROVIDER = "groq"
LLM_MODEL = "openai/gpt-oss-20b"
LLM_API_KEY = "LLM_API_KEY"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

CORPUS_PATH = "./zyro-dynamics-hr-corpus/"

In [90]:
import os
from dotenv import load_dotenv
load_dotenv()
if  os.getenv("LLM_API_KEY"):
    print("Groq api key is imported")
else:
    print("Gorq api key is not imported")
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate

print("All imports successful!")

Groq api key is imported
All imports successful!


In [91]:
CURPUS_PATH = "./zyro-dynamics-hr-corpus/"
loader = PyPDFDirectoryLoader(CURPUS_PATH)
documents = loader.load()
print(f"Loaded{len(documents)}documents")

Loaded39documents


In [92]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 100
)
chunks = splitter.split_documents(documents)
print(f"Created{len(chunks)}chunks")

Created107chunks


# Embeddings

### Hugging Face Embeddings

In [93]:
from langchain_huggingface import HuggingFaceEmbeddings

In [94]:
model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## vectorDB + Retrival
### FAISS vector DB

In [95]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks,model) # chunks :- chunks of the data, model :- Embedding models

# Retrival
retriever = vectorstore.as_retriever(search_kwargs = {"k" : 3})
print("done")

done


# LLM Initialization

In [96]:
from langchain_groq import ChatGroq
LLM_MODEL = "openai/gpt-oss-20b"
llm_model = ChatGroq(
    model=LLM_MODEL,
    temperature=0.7,
    max_tokens=500,
    api_key=LLM_API_KEY
)
print("LLM Model initialized:", LLM_MODEL)
# from langchain_groq import ChatGroq
# llm_model = ChatGroq(
# model = 'groq',
# temperature = 0.7,
# max_tokens = 500,
# api_key = LLM_API_KEY
# )
# print(f"LLM Model groq initialized")

LLM Model initialized: openai/gpt-oss-20b


In [97]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable
RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are an HR assistant.

Answer the question using ONLY the information
provided in the context.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the answer is present in the context,
   answer it clearly and directly.
4. If the answer is not present in the context,
   say:
   "I don't have that information in the provided HR documents."

Context:
{context}

Question:
{question}

Answer:
"""
)


# RAG Chain

In [98]:
from langsmith import traceable
from langchain_core.output_parsers import StrOutputParser
RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are an HR assistant.Answer the question using only
    the context below. if the answer isn't in the context,say 
    you don't have that information
    context : {context},
    Question : {question}"""
)
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)
@traceable(name="rag_chain")
def rag_chain(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    chain = RAG_PROMPT | llm_model | StrOutputParser()
    answer = chain.invoke({
        "context": context,
        "question": question
    })
    return {
        "answer": answer,
        "sources": docs
    }

In [99]:
GUARDIAL_PROMPT = ChatPromptTemplate.from_template("""
You are a scope classifier for an HR assistant.
Decide whether the question below is something an HR assistant should answer,
such as company leave policy, reimbursement, code of conduct, benefits,
working hours, attendance, or other internal HR topics.
If it is an HR-related question, respond with exactly:
IN_SCOPE
If it is not related to company HR policies, respond with exactly:
OUT_OF_SCOPE
Question: {question}
""")

REFUSAL_MESSAGE = (
    "I'm an HR assistant and can only help with questions about company HR "
    "policies (leave, reimbursement, code of conduct, etc.). I don't have "
    "information to answer that question."
)

def ask_bot(question: str):
    guardial_chain = GUARDIAL_PROMPT | llm_model | StrOutputParser()
    verdict = guardial_chain.invoke({
        "question": question
    }).strip().upper()
    print("Guardrail verdict:", verdict)
    if "OUT_OF_SCOPE" in verdict:
        return {
            "answer": REFUSAL_MESSAGE,
            "sources": []
        }
    return rag_chain(question)

print("Guardrail Initialized!")

Guardrail Initialized!


In [100]:
import pandas as pd
df = pd.read_csv("test.csv")
df

,question_id,question
0,Q01,How does my Earned Leave accrue every month?
1,Q02,How much Earned Leave can I carry forward to n...
2,Q03,How many weeks of Maternity Leave am I entitle...
3,Q04,Do I need a medical certificate for sick leave?
4,Q05,What date does my salary get credited every mo...
5,Q06,What's the salary range for an L4 Senior emplo...
6,Q07,What medical insurance coverage does the compa...
7,Q08,When am I put on a Performance Improvement Plan?
8,Q09,What's the timeline for the Annual Performance...
9,Q10,Am I eligible to work from home at my grade?


In [101]:
for i, q in enumerate(df["question"], 1):
    result = ask_bot(q)
    print(f"\nQ{i}: {q}")
    print(f"A{i}: {result['answer']}")
    sources = result.get("sources", [])
    if sources:
        print("Sources:", [
            d.metadata.get("source")
            for d in sources
        ])
    else:
        print("Sources: None")
print("-" * 60)

Guardrail verdict: IN_SCOPE

Q1: How does my Earned Leave accrue every month?
A1: Earned Leave (EL) accrues at a rate of **1.25 days per month** after you have completed one year of continuous service (provided you have worked at least 240 days that year).  

If you are still in your probation period, you accrue **0.5 days per month**, which becomes available for use only after your probation is confirmed.
Sources: ['zyro-dynamics-hr-corpus\\02_Leave_Policy.pdf', 'zyro-dynamics-hr-corpus\\02_Leave_Policy.pdf', 'zyro-dynamics-hr-corpus\\02_Leave_Policy.pdf']
Guardrail verdict: IN_SCOPE

Q2: How much Earned Leave can I carry forward to next year?
A2: I don’t have that information.
Sources: ['zyro-dynamics-hr-corpus\\02_Leave_Policy.pdf', 'zyro-dynamics-hr-corpus\\02_Leave_Policy.pdf', 'zyro-dynamics-hr-corpus\\02_Leave_Policy.pdf']
Guardrail verdict: IN_SCOPE

Q3: How many weeks of Maternity Leave am I entitled to?
A3: You’re entitled to **26 weeks of paid Maternity Leave** for your firs